# 04 — Classificadores Individuais (Tradicionais)

Este notebook treina e otimiza os **classificadores individuais** do TCC utilizando as 4 representações textuais geradas no Notebook 03.

## Classificadores abordados:
1. **Naive Bayes** — `MultinomialNB` (BoW / TF-IDF) e `GaussianNB` (Word2Vec / GloVe)
2. **Support Vector Machine (SVM)** — kernel linear e RBF
3. **Decision Tree (Árvore de Decisão)**

## Representações Textuais utilizadas:
| Arquivo | Representação |
|---|---|
| `splits_bow.joblib` | Bag of Words |
| `splits_tfidf.joblib` | TF-IDF |
| `splits_w2v.joblib` | Word2Vec |
| `splits_glove.joblib` | GloVe |

## Estratégia de Otimização:
Todos os modelos são otimizados via **GridSearchCV** com **StratifiedKFold (5 folds)**, usando `f1_weighted` como métrica de avaliação.

> **Pré-requisito:** Execute o Notebook 03 para gerar os splits em `data/processed/`.

## 1. Importações e Carregamento dos Dados

In [1]:
import sys
sys.path.append('..')  # Permite importar módulos da pasta src/ a partir de notebooks/

import os
import joblib          # Carregamento/salvamento de objetos Python (splits e modelos)
import numpy as np
import pandas as pd

from src.models import (
    criar_naive_bayes,               # Fábrica: retorna MultinomialNB ou GaussianNB conforme a representação
    criar_svm,                       # Fábrica: retorna um SVC configurado
    criar_decision_tree,             # Fábrica: retorna um DecisionTreeClassifier configurado
    PARAMS_NAIVE_BAYES_MULTINOMIAL,  # Grid de alpha para MultinomialNB
    PARAMS_NAIVE_BAYES_GAUSSIAN,     # Grid de var_smoothing para GaussianNB
    PARAMS_SVM,                      # Grid de C, kernel e gamma para SVM
    PARAMS_DECISION_TREE,            # Grid de max_depth, min_samples_split e criterion para DT
    otimizar_modelo,                 # Executa GridSearchCV com StratifiedKFold(5)
    salvar_modelo,                   # Persiste o best_estimator_ em disco (.joblib)
    para_denso,                      # Converte matriz esparsa para numpy array denso
)

os.makedirs('../results/metrics', exist_ok=True)

print('✅ Importações concluídas com sucesso!')

✅ Importações concluídas com sucesso!


In [2]:
# Carrega os 4 splits gerados no Notebook 03.
# Cada arquivo contém uma tupla: (X_train, X_test, y_train, y_test)
# Usamos '_' para descartar y_train/y_test duplicados nos splits secundários
# (o y_train e y_test são os mesmos para todos — apenas as features X mudam)
X_train_bow,   X_test_bow,   y_train, y_test = joblib.load('../data/processed/splits_bow.joblib')
X_train_tfidf, X_test_tfidf, _,       _      = joblib.load('../data/processed/splits_tfidf.joblib')
X_train_w2v,   X_test_w2v,   _,       _      = joblib.load('../data/processed/splits_w2v.joblib')
X_train_glove, X_test_glove, _,       _      = joblib.load('../data/processed/splits_glove.joblib')

print(f'BoW    — Treino: {X_train_bow.shape}   | Teste: {X_test_bow.shape}')
print(f'TF-IDF — Treino: {X_train_tfidf.shape} | Teste: {X_test_tfidf.shape}')
print(f'W2V    — Treino: {X_train_w2v.shape}   | Teste: {X_test_w2v.shape}')
print(f'GloVe  — Treino: {X_train_glove.shape} | Teste: {X_test_glove.shape}')
print(f'\nDistribuição das classes no treino (0=sem ideação | 1=com ideação):')
print(y_train.value_counts().to_string())

BoW    — Treino: (3021, 3875)   | Teste: (756, 3875)
TF-IDF — Treino: (3021, 5000) | Teste: (756, 5000)
W2V    — Treino: (3021, 100)   | Teste: (756, 100)
GloVe  — Treino: (3021, 100) | Teste: (756, 100)

Distribuição das classes no treino (0=sem ideação | 1=com ideação):
label
0    2149
1     872


## 2. Naive Bayes

O **Naive Bayes** é um classificador probabilístico baseado no **Teorema de Bayes**, que calcula a probabilidade de cada classe dado o texto de entrada. A suposição "ingênua" (naive) é que cada palavra do texto é **independente das demais**, o que simplifica drasticamente o cálculo sem perder muito desempenho.

$$P(\text{classe} \mid \text{texto}) \propto P(\text{classe}) \times \prod_{i=1}^{n} P(\text{palavra}_i \mid \text{classe})$$

**Por que duas variantes?**
| Variante | Distribuição assumida | Representações Compatíveis |
|---|---|---|
| `MultinomialNB` | Multinomial — ideal para contagens e pesos positivos | BoW, TF-IDF (valores ≥ 0) |
| `GaussianNB` | Gaussiana — modela dados contínuos com média e variância | Word2Vec, GloVe (valores negativos OK) |

**Hiperparâmetros otimizados via GridSearchCV (5-Fold Estratificado):**
- `MultinomialNB` → `alpha`: suavização de Laplace — evita probabilidade zero para palavras não vistas no treino
- `GaussianNB` → `var_smoothing`: estabilidade numérica da variância estimada por feature

### 2.1 Naive Bayes × BoW (MultinomialNB)

In [3]:
print('=' * 55)
print('  Naive Bayes (MultinomialNB) × BoW')
print('=' * 55)

# criar_naive_bayes('bow') retorna MultinomialNB(alpha=1.0) como estimador base.
# O GridSearchCV vai substituir o alpha=1.0 pelos valores do grid testados.
nb_bow = criar_naive_bayes(representacao='bow')

# otimizar_modelo executa GridSearchCV com:
#   - PARAMS_NAIVE_BAYES_MULTINOMIAL = {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]}
#   - 5-Fold Estratificado (StratifiedKFold)
#   - Métrica: f1_weighted (robusta a desbalanceamento de classes)
#   - n_jobs=-1 → usa todos os núcleos da CPU em paralelo
grid_nb_bow = otimizar_modelo(
    nb_bow,
    PARAMS_NAIVE_BAYES_MULTINOMIAL,
    X_train_bow, y_train
)

# Salva apenas o melhor estimador (melhor alpha encontrado) — não o objeto GridSearchCV inteiro
salvar_modelo(grid_nb_bow.best_estimator_, '../results/metrics/nb_bow_best.joblib')

  Naive Bayes (MultinomialNB) × BoW
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Melhores parâmetros : {'alpha': 0.01}
Melhor score (f1_weighted): 0.8450
Modelo salvo em: ../results/metrics/nb_bow_best.joblib


### 2.2 Naive Bayes × TF-IDF (MultinomialNB)

In [4]:
print('=' * 55)
print('  Naive Bayes (MultinomialNB) × TF-IDF')
print('=' * 55)

# Mesma lógica do BoW — TF-IDF também tem valores >= 0, portanto MultinomialNB é compatível.
# A diferença é que X_train_tfidf contém pesos ponderados (não apenas contagens brutas),
# o que tende a valorizar termos mais raros e discriminativos.
nb_tfidf = criar_naive_bayes(representacao='tfidf')
grid_nb_tfidf = otimizar_modelo(
    nb_tfidf,
    PARAMS_NAIVE_BAYES_MULTINOMIAL,
    X_train_tfidf, y_train
)

salvar_modelo(grid_nb_tfidf.best_estimator_, '../results/metrics/nb_tfidf_best.joblib')

  Naive Bayes (MultinomialNB) × TF-IDF
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Melhores parâmetros : {'alpha': 0.1}
Melhor score (f1_weighted): 0.8768
Modelo salvo em: ../results/metrics/nb_tfidf_best.joblib


### 2.3 Naive Bayes × Word2Vec (GaussianNB)

In [5]:
print('=' * 55)
print('  Naive Bayes (GaussianNB) × Word2Vec')
print('=' * 55)

# criar_naive_bayes('w2v') retorna GaussianNB() — sem parâmetro alpha,
# pois a distribuição Gaussiana não trabalha com contagens, mas com média e variância.
nb_w2v = criar_naive_bayes(representacao='w2v')

# para_denso(): garante que o array é denso. GaussianNB não aceita matrizes esparsas.
# PARAMS_NAIVE_BAYES_GAUSSIAN = {'var_smoothing': [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]}
grid_nb_w2v = otimizar_modelo(
    nb_w2v,
    PARAMS_NAIVE_BAYES_GAUSSIAN,
    para_denso(X_train_w2v), y_train
)

salvar_modelo(grid_nb_w2v.best_estimator_, '../results/metrics/nb_w2v_best.joblib')

  Naive Bayes (GaussianNB) × Word2Vec
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Melhores parâmetros : {'var_smoothing': 1e-11}
Melhor score (f1_weighted): 0.6912
Modelo salvo em: ../results/metrics/nb_w2v_best.joblib


### 2.4 Naive Bayes × GloVe (GaussianNB)

In [6]:
print('=' * 55)
print('  Naive Bayes (GaussianNB) × GloVe')
print('=' * 55)

# GloVe também produz vetores densos com valores negativos → GaussianNB.
# A diferença em relação ao Word2Vec é que os vetores GloVe foram pré-treinados
# em um grande corpus em português (NILC-USP), portanto capturam relações semânticas
# mais gerais da língua além do domínio específico do dataset.
nb_glove = criar_naive_bayes(representacao='glove')
grid_nb_glove = otimizar_modelo(
    nb_glove,
    PARAMS_NAIVE_BAYES_GAUSSIAN,
    para_denso(X_train_glove), y_train
)

salvar_modelo(grid_nb_glove.best_estimator_, '../results/metrics/nb_glove_best.joblib')

  Naive Bayes (GaussianNB) × GloVe
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Melhores parâmetros : {'var_smoothing': 1e-11}
Melhor score (f1_weighted): 0.7637
Modelo salvo em: ../results/metrics/nb_glove_best.joblib


## 3. Resumo dos Resultados — Naive Bayes

In [7]:
resultados_nb = pd.DataFrame([
    {'Modelo': 'Naive Bayes (MultinomialNB)', 'Representação': 'BoW',      'Melhores Params': grid_nb_bow.best_params_,   'F1-Score (CV)': round(grid_nb_bow.best_score_, 4)},
    {'Modelo': 'Naive Bayes (MultinomialNB)', 'Representação': 'TF-IDF',   'Melhores Params': grid_nb_tfidf.best_params_, 'F1-Score (CV)': round(grid_nb_tfidf.best_score_, 4)},
    {'Modelo': 'Naive Bayes (GaussianNB)',    'Representação': 'Word2Vec', 'Melhores Params': grid_nb_w2v.best_params_,   'F1-Score (CV)': round(grid_nb_w2v.best_score_, 4)},
    {'Modelo': 'Naive Bayes (GaussianNB)',    'Representação': 'GloVe',    'Melhores Params': grid_nb_glove.best_params_,  'F1-Score (CV)': round(grid_nb_glove.best_score_, 4)},
])
display(resultados_nb)
resultados_nb.to_csv('../results/metrics/resultados_nb.csv', index=False)
print('✅ resultados_nb.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,Naive Bayes (MultinomialNB),BoW,{'alpha': 0.01},0.8450
1,Naive Bayes (MultinomialNB),TF-IDF,{'alpha': 0.1},0.8768
2,Naive Bayes (GaussianNB),Word2Vec,{'var_smoothing': 1e-11},0.6912
3,Naive Bayes (GaussianNB),GloVe,{'var_smoothing': 1e-11},0.7637


✅ resultados_nb.csv salvo em results/metrics/


## 4. SVM (Support Vector Machine)

O **SVM** busca o **hiperplano de máxima margem** que separa as classes no espaço de features. Em alta dimensão (como BoW com 5.000 features), o SVM é especialmente poderoso porque a margem máxima evita overfitting mesmo com muitas variáveis.

Utilizamos o **kernel trick** para lidar com dados não linearmente separáveis:
- **Kernel Linear**: traça um hiperplano reto — eficiente para espaços de alta dimensão como BoW e TF-IDF
- **Kernel RBF** (Radial Basis Function): projeta os dados em espaço de dimensão infinita — mais flexível para embeddings densos como Word2Vec e GloVe

**Hiperparâmetros otimizados:**
| Parâmetro | O que controla |
|---|---|
| `C` | Regularização — trade-off entre margem larga (C baixo) e poucos erros no treino (C alto) |
| `kernel` | Tipo de transformação do espaço de features (`linear` ou `rbf`) |
| `gamma` | Raio de influência de cada ponto de treino (apenas kernel `rbf`) |

### 4.1 SVM × BoW

In [8]:
print('=' * 55)
print('  SVM × BoW')
print('=' * 55)

# criar_svm() retorna SVC(C=1.0, kernel='rbf', probability=True).
# probability=True é necessário para o Voting Classifier (soft voting) no Notebook 06.
# PARAMS_SVM testa 3×2×2 = 12 combinações × 5 folds = 60 treinos.
# O kernel linear tende a ser mais rápido e eficaz para BoW de alta dimensão.
svm_bow = criar_svm()
grid_svm_bow = otimizar_modelo(
    svm_bow,
    PARAMS_SVM,
    X_train_bow, y_train
)
salvar_modelo(grid_svm_bow.best_estimator_, '../results/metrics/svm_bow_best.joblib')

  SVM × BoW
Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\Patrick\facul\TCC\tcc-ideacao-suicida\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Melhores parâmetros : {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Melhor score (f1_weighted): 0.8827
Modelo salvo em: ../results/metrics/svm_bow_best.joblib


### 4.2 SVM × TF-IDF

In [9]:
print('=' * 55)
print('  SVM × TF-IDF')
print('=' * 55)

# TF-IDF é esparso de alta dimensão. O kernel linear costuma superar o RBF
# nesse cenário por ter complexidade O(n_features) em vez de O(n_samples²).
svm_tfidf = criar_svm()
grid_svm_tfidf = otimizar_modelo(
    svm_tfidf,
    PARAMS_SVM,
    X_train_tfidf, y_train
)
salvar_modelo(grid_svm_tfidf.best_estimator_, '../results/metrics/svm_tfidf_best.joblib')

  SVM × TF-IDF
Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\Patrick\facul\TCC\tcc-ideacao-suicida\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Melhores parâmetros : {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Melhor score (f1_weighted): 0.8933
Modelo salvo em: ../results/metrics/svm_tfidf_best.joblib


### 4.3 SVM × Word2Vec

In [10]:
print('=' * 55)
print('  SVM × Word2Vec')
print('=' * 55)

# Word2Vec é denso de baixa dimensão (100d). O kernel RBF tende a capturar
# melhor as fronteiras não-lineares no espaço de embeddings.
svm_w2v = criar_svm()
grid_svm_w2v = otimizar_modelo(
    svm_w2v,
    PARAMS_SVM,
    para_denso(X_train_w2v), y_train
)
salvar_modelo(grid_svm_w2v.best_estimator_, '../results/metrics/svm_w2v_best.joblib')

  SVM × Word2Vec
Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\Patrick\facul\TCC\tcc-ideacao-suicida\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Melhores parâmetros : {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Melhor score (f1_weighted): 0.7940
Modelo salvo em: ../results/metrics/svm_w2v_best.joblib


### 4.4 SVM × GloVe

In [11]:
print('=' * 55)
print('  SVM × GloVe')
print('=' * 55)

# GloVe pré-treinado no corpus NILC-USP: vetores que já codificam semântica
# do português de forma mais rica. O SVM com kernel RBF pode explorar
# essas relações semânticas na geometria do espaço de 100 dimensões.
svm_glove = criar_svm()
grid_svm_glove = otimizar_modelo(
    svm_glove,
    PARAMS_SVM,
    para_denso(X_train_glove), y_train
)
salvar_modelo(grid_svm_glove.best_estimator_, '../results/metrics/svm_glove_best.joblib')

  SVM × GloVe
Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\Patrick\facul\TCC\tcc-ideacao-suicida\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Melhores parâmetros : {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Melhor score (f1_weighted): 0.8490
Modelo salvo em: ../results/metrics/svm_glove_best.joblib


## 5. Resumo dos Resultados — SVM

In [12]:
resultados_svm = pd.DataFrame([
    {'Modelo': 'SVM', 'Representação': 'BoW',      'Melhores Params': grid_svm_bow.best_params_,   'F1-Score (CV)': round(grid_svm_bow.best_score_, 4)},
    {'Modelo': 'SVM', 'Representação': 'TF-IDF',   'Melhores Params': grid_svm_tfidf.best_params_, 'F1-Score (CV)': round(grid_svm_tfidf.best_score_, 4)},
    {'Modelo': 'SVM', 'Representação': 'Word2Vec', 'Melhores Params': grid_svm_w2v.best_params_,   'F1-Score (CV)': round(grid_svm_w2v.best_score_, 4)},
    {'Modelo': 'SVM', 'Representação': 'GloVe',    'Melhores Params': grid_svm_glove.best_params_,  'F1-Score (CV)': round(grid_svm_glove.best_score_, 4)},
])
display(resultados_svm)
resultados_svm.to_csv('../results/metrics/resultados_svm.csv', index=False)
print('✅ resultados_svm.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,SVM,BoW,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.8827
1,SVM,TF-IDF,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.8933
2,SVM,Word2Vec,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.7940
3,SVM,GloVe,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.8490


✅ resultados_svm.csv salvo em results/metrics/


## 6. Decision Tree (Árvore de Decisão)

A **Árvore de Decisão** divide recursivamente o espaço de features criando regras do tipo *"se palavra X aparece mais de N vezes, então classe Y"*. É o único classificador individualmente interpretável: é possível visualizar o caminho exato de decisão para cada texto.

**Conceitos-chave:**
- **Critério de divisão**: como medir a impureza de um nó — `gini` ou `entropy` (ganho de informação)
- **Profundidade máxima (`max_depth`)**: limita o crescimento da árvore para evitar overfitting
- **Amostras mínimas por divisão (`min_samples_split`)**: um nó só divide se tiver pelo menos N amostras

**Hiperparâmetros otimizados via GridSearchCV:**
```
PARAMS_DECISION_TREE = {
    'max_depth': [None, 5, 10, 20],      # None = cresce até folhas puras
    'min_samples_split': [2, 5, 10],     # Mínimo de amostras para dividir um nó
    'criterion': ['gini', 'entropy'],    # Métrica de impureza
}
# Total: 4 × 3 × 2 = 24 combinações × 5 folds = 120 treinos
```

### 6.1 Decision Tree × BoW

In [13]:
print('=' * 55)
print('  Decision Tree × BoW')
print('=' * 55)

# Com BoW de 5.000 features, a árvore sem limite de profundidade cresce muito
# e tende a overfittar (memoriza o treino). O GridSearch vai encontrar o
# max_depth ideal que equilibra bias e variância.
# A métrica gini é geralmente mais rápida computacionalmente que entropy.
dt_bow = criar_decision_tree()
grid_dt_bow = otimizar_modelo(
    dt_bow,
    PARAMS_DECISION_TREE,
    X_train_bow, y_train
)
salvar_modelo(grid_dt_bow.best_estimator_, '../results/metrics/dt_bow_best.joblib')

  Decision Tree × BoW
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Melhores parâmetros : {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2}
Melhor score (f1_weighted): 0.8292
Modelo salvo em: ../results/metrics/dt_bow_best.joblib


### 6.2 Decision Tree × TF-IDF

In [14]:
print('=' * 55)
print('  Decision Tree × TF-IDF')
print('=' * 55)

# Com TF-IDF, os valores são contínuos e ponderados. O critério entropy
# (ganho de informação) pode se beneficiar dessa riqueza de valores
# para escolher melhores pontos de corte nos nós.
dt_tfidf = criar_decision_tree()
grid_dt_tfidf = otimizar_modelo(
    dt_tfidf,
    PARAMS_DECISION_TREE,
    X_train_tfidf, y_train
)
salvar_modelo(grid_dt_tfidf.best_estimator_, '../results/metrics/dt_tfidf_best.joblib')

  Decision Tree × TF-IDF
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Melhores parâmetros : {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2}
Melhor score (f1_weighted): 0.8421
Modelo salvo em: ../results/metrics/dt_tfidf_best.joblib


### 6.3 Decision Tree × Word2Vec

In [15]:
print('=' * 55)
print('  Decision Tree × Word2Vec')
print('=' * 55)

# Word2Vec produz apenas 100 features (uma por dimensão do embedding médio).
# Árvores geralmente performam melhor com menos features, pois há menos risco
# de splits irrelevantes. A interpretabilidade também melhora: cada nó
# representa uma dimensão semântica do espaço de embeddings.
dt_w2v = criar_decision_tree()
grid_dt_w2v = otimizar_modelo(
    dt_w2v,
    PARAMS_DECISION_TREE,
    para_denso(X_train_w2v), y_train
)
salvar_modelo(grid_dt_w2v.best_estimator_, '../results/metrics/dt_w2v_best.joblib')

  Decision Tree × Word2Vec
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Melhores parâmetros : {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 2}
Melhor score (f1_weighted): 0.8044
Modelo salvo em: ../results/metrics/dt_w2v_best.joblib


### 6.4 Decision Tree × GloVe

In [16]:
print('=' * 55)
print('  Decision Tree × GloVe')
print('=' * 55)

# GloVe pré-treinado NILC-USP (100d). Similar ao Word2Vec, mas os vetores
# codificam co-ocorrências globais do corpus — o que pode criar dimensões
# mais estáveis para os splits da árvore.
dt_glove = criar_decision_tree()
grid_dt_glove = otimizar_modelo(
    dt_glove,
    PARAMS_DECISION_TREE,
    para_denso(X_train_glove), y_train
)
salvar_modelo(grid_dt_glove.best_estimator_, '../results/metrics/dt_glove_best.joblib')

  Decision Tree × GloVe
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Melhores parâmetros : {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 5}
Melhor score (f1_weighted): 0.8056
Modelo salvo em: ../results/metrics/dt_glove_best.joblib


## 7. Resumo dos Resultados — Decision Tree

In [17]:
resultados_dt = pd.DataFrame([
    {'Modelo': 'Decision Tree', 'Representação': 'BoW',      'Melhores Params': grid_dt_bow.best_params_,   'F1-Score (CV)': round(grid_dt_bow.best_score_, 4)},
    {'Modelo': 'Decision Tree', 'Representação': 'TF-IDF',   'Melhores Params': grid_dt_tfidf.best_params_, 'F1-Score (CV)': round(grid_dt_tfidf.best_score_, 4)},
    {'Modelo': 'Decision Tree', 'Representação': 'Word2Vec', 'Melhores Params': grid_dt_w2v.best_params_,   'F1-Score (CV)': round(grid_dt_w2v.best_score_, 4)},
    {'Modelo': 'Decision Tree', 'Representação': 'GloVe',    'Melhores Params': grid_dt_glove.best_params_,  'F1-Score (CV)': round(grid_dt_glove.best_score_, 4)},
])
display(resultados_dt)
resultados_dt.to_csv('../results/metrics/resultados_dt.csv', index=False)
print('✅ resultados_dt.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,Decision Tree,BoW,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.8292
1,Decision Tree,TF-IDF,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.8421
2,Decision Tree,Word2Vec,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.8044
3,Decision Tree,GloVe,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.8056


✅ resultados_dt.csv salvo em results/metrics/


## 8. Resumo Geral — Todos os Classificadores Individuais

In [18]:
# Consolida os resultados dos 3 classificadores individuais em uma única tabela.
# Lê os CSVs individuais para funcionar mesmo que as células acima não
# tenham sido executadas nesta sessão (útil para revisão posterior).
df_nb  = pd.read_csv('../results/metrics/resultados_nb.csv')
df_svm = pd.read_csv('../results/metrics/resultados_svm.csv')
df_dt  = pd.read_csv('../results/metrics/resultados_dt.csv')

resumo_geral = pd.concat([df_nb, df_svm, df_dt], ignore_index=True)

# Ordena da melhor para a pior combinação
resumo_geral = resumo_geral.sort_values('F1-Score (CV)', ascending=False).reset_index(drop=True)

display(resumo_geral)

resumo_geral.to_csv('../results/metrics/resultados_individuais.csv', index=False)
print('✅ resultados_individuais.csv salvo em results/metrics/')
print(f'\nMelhor combinação encontrada:')
print(f"  {resumo_geral.iloc[0]['Modelo']} × {resumo_geral.iloc[0]['Representação']}")
print(f"  F1-Score (CV): {resumo_geral.iloc[0]['F1-Score (CV)']}")

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,SVM,TF-IDF,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.8933
1,SVM,BoW,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.8827
2,Naive Bayes (MultinomialNB),TF-IDF,{'alpha': 0.1},0.8768
3,SVM,GloVe,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.8490
4,Naive Bayes (MultinomialNB),BoW,{'alpha': 0.01},0.8450
5,Decision Tree,TF-IDF,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.8421
6,Decision Tree,BoW,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.8292
7,Decision Tree,GloVe,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.8056
8,Decision Tree,Word2Vec,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.8044
9,SVM,Word2Vec,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.7940


✅ resultados_individuais.csv salvo em results/metrics/

Melhor combinação encontrada:
  SVM × TF-IDF
  F1-Score (CV): 0.8933
